In [90]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [91]:
import pandas as pd
import re
from collections import Counter

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.linear_model import PassiveAggressiveClassifier, LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, f1_score
from sklearn.feature_extraction.text import CountVectorizer


In [92]:
custom_stopwords = {
    "said", "reuters", "will", "new",
    "featured", "image", "images",
    "twitter", "com", "pic",
    "getty", "screen", "capture"
}

stop_words = set(ENGLISH_STOP_WORDS).union(custom_stopwords)

In [93]:


fake_yolu = '/content/drive/MyDrive/Colab_Notebooks/Fake.csv'
true_yolu = '/content/drive/MyDrive/Colab_Notebooks/True.csv'


df_fake = pd.read_csv(fake_yolu)
df_true = pd.read_csv(true_yolu)

# print(f"Gerçek haber sayısı: {len(df_true)}")
# print(f"Sahte haber sayısı: {len(df_fake)}")


In [94]:
df_fake['label'] = 0
df_true['label'] = 1

df = pd.concat([df_fake, df_true], axis=0).reset_index(drop=True)



In [95]:
def clean_text(text):
    text = text.lower()

    text = re.sub(r'\[.*?\]', '', text)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'\W', ' ', text)

    text = re.sub(r'\s+', ' ', text).strip()

    return text


df['text'] = df['text'].apply(clean_text)

In [96]:


df = df.drop_duplicates(subset='text')

In [97]:
df["word_count"] = df["text"].apply(lambda x: len(x.split()))

print("Ortalama kelime sayısı:")
print(df.groupby("label")["word_count"].mean())

print("\nMedyan kelime sayısı:")
print(df.groupby("label")["word_count"].median())

Ortalama kelime sayısı:
label
0    438.457164
1    392.990823
Name: word_count, dtype: float64

Medyan kelime sayısı:
label
0    387.0
1    366.0
Name: word_count, dtype: float64


In [98]:
x_train, x_test, y_train, y_test = train_test_split(
    df['text'],
    df['label'],
    test_size=0.2,
    random_state=7
)

In [99]:
train_texts = set(x_train)
test_texts = set(x_test)

common = train_texts.intersection(test_texts)
print("Ortak veri sayısı:", len(common))

Ortak veri sayısı: 0


In [100]:
tfidf_vectorizer = TfidfVectorizer(
    stop_words=list(stop_words),
    max_df=0.7,
    ngram_range=(1,2)
)

tfidf_train = tfidf_vectorizer.fit_transform(x_train)
tfidf_test = tfidf_vectorizer.transform(x_test)

In [101]:
models = {
    "Passive Aggressive": PassiveAggressiveClassifier(max_iter=50, random_state=7),
    "Naive Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=7)
}

print("Model Karşılaştırması:\n")

for name, model in models.items():
    model.fit(tfidf_train, y_train)
    pred = model.predict(tfidf_test)

    print(name)
    print("Accuracy:", accuracy_score(y_test, pred))
    print("F1 Score:", f1_score(y_test, pred))
    print("-----------------------------")

Model Karşılaştırması:

Passive Aggressive
Accuracy: 0.9836878507112097
F1 Score: 0.9852333136444182
-----------------------------
Naive Bayes
Accuracy: 0.920657705859324
F1 Score: 0.9319910514541387
-----------------------------
Logistic Regression
Accuracy: 0.9697246509200053
F1 Score: 0.9726801695713613
-----------------------------


In [102]:
pac = PassiveAggressiveClassifier(max_iter=50, random_state=7)
pac.fit(tfidf_train, y_train)
y_pred = pac.predict(tfidf_test)
score = accuracy_score(y_test, y_pred)
print(f'Doğruluk Oranı: %{round(score * 100, 2)}')

Doğruluk Oranı: %98.37


In [103]:

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=["Fake News", "True News"]))

Confusion Matrix:
[[3368   81]
 [  44 4170]]
Classification Report:
              precision    recall  f1-score   support

   Fake News       0.99      0.98      0.98      3449
   True News       0.98      0.99      0.99      4214

    accuracy                           0.98      7663
   macro avg       0.98      0.98      0.98      7663
weighted avg       0.98      0.98      0.98      7663



In [104]:
def predict_news(news):
    cleaned_news = clean_text(news)
    vectorized_news = tfidf_vectorizer.transform([cleaned_news])
    prediction = pac.predict(vectorized_news)

    if prediction[0] == 1:
        return "TRUE NEWS"
    else:
        return "FAKE NEWS"

In [105]:
sample_news_1 = "Breaking news! Scientists confirm that eating one banana every day will make you live forever. Click here now!"
print(predict_news(sample_news_1))

sample_news_2 = "The president met with foreign leaders today to discuss economic cooperation and international trade agreements."
print(predict_news(sample_news_2))

sample_news_3 = """
The White House announced on Monday that officials will meet with European leaders
to discuss trade relations, economic cooperation, and international security issues.
The meeting is expected to take place later this week, according to government sources.
"""

print(predict_news(sample_news_3))

sample_news_4 = """
The government announced new economic reforms aimed at reducing inflation and improving
foreign investment opportunities in the country.
"""
print(predict_news(sample_news_4))

FAKE NEWS
TRUE NEWS
TRUE NEWS
TRUE NEWS


In [106]:

fake_text = " ".join(df[df["label"] == 0]["text"])
true_text = " ".join(df[df["label"] == 1]["text"])


fake_words = fake_text.lower().split()
true_words = true_text.lower().split()



fake_words_clean = [w for w in fake_words if w not in stop_words and len(w) > 2]
true_words_clean = [w for w in true_words if w not in stop_words and len(w) > 2]


fake_counts = Counter(fake_words_clean)
true_counts = Counter(true_words_clean)


print("Fake News - Temizlenmiş en sık kelimeler:\n")
for word, count in fake_counts.most_common(20):
    print(f"{word}: {count}")

print("-----------------------------")

print("True News - Temizlenmiş en sık kelimeler:\n")
for word, count in true_counts.most_common(20):
    print(f"{word}: {count}")

Fake News - Temizlenmiş en sık kelimeler:

trump: 67937
people: 21157
president: 20751
just: 16732
donald: 15271
like: 14348
clinton: 13481
obama: 13344
time: 10700
white: 10284
news: 10195
hillary: 9910
state: 9470
right: 8882
campaign: 8781
house: 8411
know: 8402
don: 8191
american: 8173
america: 8143
-----------------------------
True News - Temizlenmiş en sık kelimeler:

trump: 52946
president: 27360
state: 20504
government: 18363
house: 16278
republican: 16064
states: 15999
people: 14951
united: 14884
year: 14452
told: 13843
washington: 12607
party: 12535
election: 12048
campaign: 10455
donald: 10163
percent: 9883
security: 9857
clinton: 9436
white: 9328


In [107]:
def get_top_ngrams(texts, ngram_range=(2,2), n=20):
    vectorizer = CountVectorizer(
        stop_words=list(stop_words),
        ngram_range=ngram_range
    )
    X = vectorizer.fit_transform(texts)
    counts = X.sum(axis=0).A1
    words = vectorizer.get_feature_names_out()
    result = sorted(zip(words, counts), key=lambda x: x[1], reverse=True)
    return result[:n]

print("\n=============================\n")

print("Fake News - En sık bigramlar:")
for word, count in get_top_ngrams(df[df["label"] == 0]["text"], (2,2), 20):
    print(f"{word}: {count}")

print("\n-----------------------------\n")

print("True News - En sık bigramlar:")
for word, count in get_top_ngrams(df[df["label"] == 1]["text"], (2,2), 20):
    print(f"{word}: {count}")



Fake News - En sık bigramlar:
donald trump: 14012
hillary clinton: 5416
white house: 5188
united states: 4948
president obama: 3448
fox news: 2675
president trump: 2610
year old: 2009
trump campaign: 1631
barack obama: 1566
trump realdonaldtrump: 1535
supreme court: 1451
republican party: 1358
ted cruz: 1340
fake news: 1275
2017 realdonaldtrump: 1237
american people: 1232
social media: 1224
national security: 1175
21st century: 1169

-----------------------------

True News - En sık bigramlar:
united states: 11623
donald trump: 9878
white house: 8204
president donald: 5671
north korea: 5129
prime minister: 4055
islamic state: 3399
barack obama: 3302
told reporters: 3049
president barack: 2928
hillary clinton: 2473
supreme court: 2406
trump administration: 2352
house representatives: 2222
united nations: 2211
secretary state: 2182
year old: 2167
national security: 2086
human rights: 2025
european union: 1902
